# Docker Compose: Multi-Container Projects


## Why Docker Compose?

Real backend projects usually need more than one container.

Example Django project:

```text
web container       → Django/DRF app
db container        → PostgreSQL
redis container     → Redis cache / Celery broker
celery container    → background tasks
nginx container     → reverse proxy / static files
```

So Dockerizing an app does not always mean:

```text
one project = one container
```

More realistic:

```text
one project = multiple containers working together
```

Running each container manually with `docker run` becomes hard.

Docker Compose lets us describe all services in one YAML file and run them together.


## Compose File Structure

Recommended modern file name:

```text
compose.yaml
```

Older projects often use:

```text
docker-compose.yml
```

Basic structure:

```yaml
services:
  app:
    image: my_python_app
    ports:
      - "5000:5000"
```

Run:

```bash
docker compose up -d
```

Older command style:

```bash
docker-compose up -d
```

In this course, we use the modern `docker compose` command.


## Common Compose Concepts

| Concept | Meaning |
|---------|---------|
| `services` | Containers that make up the project. |
| `image` | Use an existing image. |
| `build` | Build an image from a Dockerfile. |
| `ports` | Publish container ports to the host. |
| `environment` | Set environment variables. |
| `env_file` | Load variables from a file. |
| `volumes` | Persist or share data. |
| `depends_on` | Start one service before another. |
| `networks` | Control how services communicate. |

Compose creates a default network. Services can reach each other by service name.


## Basic Commands

Start services:

```bash
docker compose up -d
```

View logs:

```bash
docker compose logs -f
```

View service status:

```bash
docker compose ps
```

Run a command inside a service:

```bash
docker compose exec app bash
```

Stop and remove containers/network:

```bash
docker compose down
```

Stop and remove containers/network/volumes:

```bash
docker compose down -v
```

Be careful: `down -v` removes volumes and can delete database data.


## Django + PostgreSQL Example

```yaml
services:
  web:
    build: .
    ports:
      - "8000:8000"
    depends_on:
      - db
    environment:
      DJANGO_DEBUG: "True"
      DATABASE_URL: postgres://user:password@db:5432/mydatabase
    command: python manage.py runserver 0.0.0.0:8000

  db:
    image: postgres:15
    environment:
      POSTGRES_USER: user
      POSTGRES_PASSWORD: password
      POSTGRES_DB: mydatabase
```

Important part:

```text
DATABASE_URL=postgres://user:password@db:5432/mydatabase
```

The host is `db`, not `localhost`, because `db` is the Compose service name.


## Why `localhost` is Wrong Between Containers

Inside the Django container:

```text
localhost = the Django container itself
```

PostgreSQL is running in another container, so Django should connect to:

```text
db:5432
```

because the database service is named `db`.

Mental model:

```text
web container ── connects to hostname db ──► db container
```


## Running Django Commands

Start services:

```bash
docker compose up -d
```

Run migrations:

```bash
docker compose exec web python manage.py migrate
```

Create superuser:

```bash
docker compose exec web python manage.py createsuperuser
```

Open Django shell:

```bash
docker compose exec web python manage.py shell
```

View web logs:

```bash
docker compose logs -f web
```


## Volumes and Data Persistence

Without a volume, database data may disappear when containers are removed.

Add a named volume:

```yaml
services:
  db:
    image: postgres:15
    volumes:
      - pgdata:/var/lib/postgresql/data

volumes:
  pgdata:
```

View volumes:

```bash
docker volume ls
```

Remove a volume:

```bash
docker volume rm VOLUME_NAME
```

Warning:

```bash
docker compose down -v
```

removes Compose volumes too. This can delete your database data.


## Environment Variables and `.env`

Compose can read variables from a `.env` file.

`.env`:

```dotenv
POSTGRES_USER=user
POSTGRES_PASSWORD=password
POSTGRES_DB=mydatabase
DJANGO_DEBUG=True
```

`compose.yaml`:

```yaml
services:
  db:
    image: postgres:15
    environment:
      POSTGRES_USER: ${POSTGRES_USER}
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD}
      POSTGRES_DB: ${POSTGRES_DB}
```

Best practice:

- Commit `.env.example`.
- Do not commit real `.env` files.
- Use different values for development and production.


## Scaling Services

Compose can start multiple containers for one service:

```bash
docker compose up --scale web=3 -d
```

View status:

```bash
docker compose ps
```

Beginner note:

Scaling is not just “run more containers.” You also need load balancing and careful settings. For now, know that Compose can scale services, but we will not go deep.


## Curious Note: Beyond Compose

Docker Compose is good for learning, local development, and simple single-server deployments.

Big companies may use orchestration tools when they have many containers across many servers.

Examples:

- Kubernetes
- Docker Swarm
- cloud container platforms

These tools help with scaling, rolling updates, self-healing, and load balancing.

For this course, just remember the name **Kubernetes** as the advanced next step. Docker Compose is enough for now.


## Summary

- Docker Compose manages multi-container projects.
- Services can communicate by service name.
- Use volumes for persistent data such as PostgreSQL files.
- Use `docker compose exec` to run commands inside services.
- `docker compose down -v` can delete data volumes.
- Compose is a step before orchestration tools such as Kubernetes.
